# IBM Telco Customer Churn — Exploratory Data Analysis
Data are observational; no plot establishes causes of churn. Run from the repository root after installation and download.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from churn_prediction.config import DATA, REPORTS
from churn_prediction.data_validation import load_data
df = load_data(); df.shape

In [ ]:
display(df.head(), df.dtypes.to_frame("dtype"), df.describe(include="all").T)
print("Null counts:",df.isna().sum().to_dict())
print("Duplicate IDs:",df.customerID.duplicated().sum(),"Full duplicates:",df.duplicated().sum())

`Churn == Yes` is the label. `customerID` is an identifier, not a predictor. `TotalCharges` blanks become NaN (imputed during model fitting). Historical account totals may leak information if unavailable at prospective prediction time.

In [ ]:
df["churn_binary"] = df.Churn.eq("Yes").astype(int)
fig,ax=plt.subplots(figsize=(6,4));sns.countplot(data=df,x="Churn",ax=ax,color="steelblue");ax.set_title("Observed churn class counts");plt.show()
print("Churn prevalence:",df.churn_binary.mean())

In [ ]:
df["tenure_band"] = pd.cut(df.tenure,[-.01,12,24,48,np.inf],labels=["0–12","13–24","25–48","49+"])
df["charge_band"] = pd.qcut(df.MonthlyCharges,q=4,duplicates="drop")
fig,axes=plt.subplots(3,2,figsize=(13,13))
for ax,col in zip(axes.ravel(),["tenure_band","Contract","PaymentMethod","charge_band","InternetService"]):
    df.groupby(col,observed=True).churn_binary.mean().plot.bar(ax=ax)
    ax.set(title=f"Observed churn rate by {col}",xlabel=col,ylabel="Churn fraction",ylim=(0,1));ax.tick_params(axis="x",rotation=30)
axes.ravel()[-1].axis("off");plt.tight_layout();plt.show()

Interpretation: compare heights and sample sizes. Large group rate differences are associations; not proof that switching contract/payment/services changes churn.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,col in zip(axes,["tenure","MonthlyCharges","TotalCharges"]):
    sns.histplot(data=df,x=col,hue="Churn",element="step",stat="density",common_norm=False,ax=ax);ax.set_title(col)
plt.tight_layout();plt.show()
fig,ax=plt.subplots(figsize=(6,4));sns.heatmap(df[["tenure","MonthlyCharges","TotalCharges","SeniorCitizen","churn_binary"]].corr(),annot=True,ax=ax,cmap="coolwarm",vmin=-1,vmax=1);plt.show()

**H0**: independence for each categorical variable versus churn; identical numerical rank distributions for each two-group comparison. **H1**: association or differing distributions. Chi-square assumes independent records and adequate expected cell counts; Mann–Whitney assumes independent observations. Benjamini–Hochberg FDR corrects multiple p-values. Cramér’s V and rank-biserial quantify effect sizes, not causal treatment impact.

In [ ]:
tests=[]
for col in ["Contract","PaymentMethod","InternetService","Partner","PaperlessBilling"]:
    tbl=pd.crosstab(df[col],df.Churn); chi,p,_,expected=chi2_contingency(tbl)
    tests.append((col,"Chi-square",p,np.sqrt(chi/(tbl.to_numpy().sum()*min(tbl.shape[0]-1,tbl.shape[1]-1))),expected.min()))
for col in ["tenure","MonthlyCharges","TotalCharges"]:
    a=df.loc[df.churn_binary==1,col].dropna();b=df.loc[df.churn_binary==0,col].dropna();stat,p=mannwhitneyu(a,b,alternative="two-sided")
    tests.append((col,"Mann-Whitney U",p,2*stat/(len(a)*len(b))-1,np.nan))
results=pd.DataFrame(tests,columns=["feature","test","p_value","effect_size","min_expected_cell"]);results["FDR_BH_p"]=multipletests(results.p_value,method="fdr_bh")[1]
display(results.sort_values("FDR_BH_p"))

Use effect magnitudes and domain knowledge, not p-value alone. Test significance is conditional on the sample and test assumptions. A future deployable churn model needs time-stamped features and temporal validation.

In [ ]:
from churn_prediction.eda import run_eda
summary=run_eda()
print("Saved figures, test table and interpretation summary in",REPORTS)